# 实验四 · 三角形负载 —— 循环调度与负载均衡

**所属**：《并行计算技术》第五章 · OpenMP 编程　|　**难度**：⭐⭐ 基础　|　**预计时长**：30–40 分钟

实验二与实验三解决了并行程序的正确性问题，本实验起讨论转向性能。第一个性能议题是**负载不均衡**：当各次迭代的工作量不相等时，默认的均匀划分会使部分线程提前完成，并在隐式栅栏处等待。本实验以工作量随下标线性增长的循环为载体，同时展示四种调度类型的划分方式与相应开销。

> **实验说明**
> 1. 本实验采用**递进式的版本组织**：以串行实现为基准，每个版本仅引入一种新的 OpenMP 构造或一处相应的代码改写，并在同一次运行中完成全部版本的计时与正确性校验，因而各版本面对的是完全相同的数据与运行环境，各版本之间具有可比性。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖支持 OpenMP 的 **GCC 编译器**，建议在华为鲲鹏处理器或其他 AArch64 平台上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 本实验只有**一个**计算内核。表格中的各行不是不同的算法，而是同一个内核在不同调度配置下的表现。
> 6. 程序在 `n <= 64` 时会额外打印「迭代到线程」的字符映射图。该图是理解四种调度类型划分方式的直观材料，建议仔细观察。

## 🎯 学习目标

完成本实验后，学生应能够：

- 说明**负载不均衡**的成因，并解释为何隐式栅栏会把不均衡转化为整体的性能损失
- 掌握 `schedule` 子句的四种调度类型：`static`、`dynamic`、`guided`、`auto`，以及 `chunk_size` 参数的含义
- 能够根据迭代工作量的分布特征，为一个具体循环选择合适的调度类型与块大小
- 理解**调度开销**与**均衡收益**之间的取舍：块越小越均衡，但分配次数越多
- 掌握 `schedule(runtime)` 与 `omp_set_schedule()` / `OMP_SCHEDULE` 的配合方式，实现「一次编译、多种策略」的对照实验
- 了解线程亲和性 `OMP_PROC_BIND` 与 `OMP_PLACES` 的作用，以及它们在异构多核平台上的必要性

## 🗺️ 学习路径

1. **准备阶段**：分析本实验负载的工作量分布，估算不均衡的严重程度
2. **串行基准**：确立参照
3. **五种配置对照**：在同一次运行中依次切换 `static` / `static, chunk` / `dynamic, chunk` / `guided` / `auto` 五种调度配置
4. **映射图观察**：以小规模问题打印迭代到线程的归属，直观比较划分方式
5. **可视化与分析**：绘制加速比柱状图，解释各调度类型的适用场合
6. **扩展实验**：块大小扫描，寻找调度开销与均衡收益的平衡点

## 1. 背景知识：负载不均衡

### 1.1 问题的由来

`#pragma omp for` 的默认行为是把迭代空间**均匀划分为连续块**：$n$ 次迭代分给 $p$ 个线程，每个线程连续承担约 $n/p$ 次。

当每次迭代的工作量大致相等时，这一划分是理想的。但许多实际问题并非如此：

| 场景 | 工作量分布 |
|---|---|
| 三角矩阵运算 | 第 $i$ 行的元素数随 $i$ 线性变化 |
| 稀疏矩阵按行处理 | 各行非零元个数差异悬殊 |
| 迭代法求解，逐点收敛 | 收敛快的点提前退出 |
| 图像处理，按区域检测 | 目标密集区耗时远高于空白区 |
| 蒙特卡洛，带拒绝采样 | 每个样本的接受次数不定 |

### 1.2 隐式栅栏把不均衡转化为损失

实验一已经指出，并行区域出口存在隐式栅栏。这意味着**整个并行区域的执行时间由执行时间最长的线程决定**：

```text
  线程0 ████                          ← 提前完成，进入等待
  线程1 ████████
  线程2 ████████████
  线程3 ████████████████████          ← 决定整体耗时
        └──────── 实际耗时 ────────┘
        └─ 有效计算 ─┘└── 空等 ──┘
```

设各线程的工作量为 $W_0, W_1, \ldots, W_{p-1}$，则并行耗时正比于 $\max_t W_t$，而理想耗时正比于 $\frac{1}{p}\sum_t W_t$。两者之比即为负载不均衡条件下的**并行效率**，其与 1 的差值即为不均衡造成的效率损失：

$$\text{效率} = \frac{\frac{1}{p}\sum_t W_t}{\max_t W_t}$$

### 1.3 本实验的负载模型

本实验构造了一个工作量精确可控的负载：第 $i$ 次迭代执行 $i+1$ 次 `sin()` 调用。

```c
static double f(long i) {
  long first = i * (i + 1) / 2;
  long last = first + i;
  double val = 0.0;
  for (long j = first; j <= last; j++) {
    val += sin((double)j);      // 共 i+1 次
  }
  return val;
}
```

总工作量为 $\sum_{i=0}^{n-1}(i+1) = \frac{n(n+1)}{2}$，呈**三角形分布**。

以 $n$ 次迭代、$p$ 个线程、默认块划分为例，最后一个线程承担的工作量约为：

$$W_{p-1} \approx \frac{n^2}{2}\left[1 - \left(\frac{p-1}{p}\right)^2\right], \qquad \bar{W} = \frac{1}{p}\cdot\frac{n^2}{2}$$

$$\Longrightarrow\quad \frac{W_{p-1}}{\bar{W}} = p\left[1 - \left(\frac{p-1}{p}\right)^2\right] = \frac{2p-1}{p}$$

当 $p = 4$ 时该比值为 $1.75$，即工作量最重的线程其负载是平均值的 1.75 倍，**理论加速比上限约为 $4 / 1.75 \approx 2.3$**，而非 4。这是默认 `static` 在本实验中加速比明显偏低的主要原因。

## 2. 调度类型与调度配置

### 2.1 语法

```c
#pragma omp parallel for schedule(kind [, chunk_size])
```

### 2.2 调度类型对照

OpenMP 定义了四种调度类型：`static`、`dynamic`、`guided`、`auto`（另有 `runtime`，用于把决定推迟到运行时，见 3.1 节）。其中 `static` 按是否指定 `chunk_size` 分为两种配置，因此本实验共对照**五种调度配置**。

| 调度配置 | 划分方式 | 分配时机 | 调度开销 | 适用场合 |
|---|---|---|---|---|
| `static` | 均匀划分为 $p$ 个连续大块 | 进入循环时确定 | 最低 | 各迭代工作量相近 |
| `static, c` | 按块长 $c$ 轮转分配 | 进入时确定 | 很低 | 工作量单调变化 |
| `dynamic, c` | 先到先得，每次取 $c$ 个 | 运行时动态 | 较高 | 工作量随机且差异大 |
| `guided` | 块长由大到小递减 | 运行时动态 | 中等 | 工作量差异大，兼顾开销 |
| `auto` | 交由编译器/运行时决定 | 实现相关 | 实现相关 | 交由实现决定，不宜作为性能优化手段 |

### 2.3 划分方式示意（$n=16$，$p=4$）

```text
  迭代:      0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15

  static     0  0  0  0  1  1  1  1  2  2  2  2  3  3  3  3
                └─ 连续大块，线程 3 承担工作量最重的尾部 ─┘

  static,2   0  0  1  1  2  2  3  3  0  0  1  1  2  2  3  3
                └─ 轮转，重活与轻活被均匀摊到各线程 ─┘

  dynamic,2  谁先空闲谁先取，映射不确定

  guided     块长递减：4 4 3 2 1 1 1 ...（先粗后细）
```

### 2.4 各策略的取舍

**`static`**：运行期开销最低。各线程在进入循环时即已确定自己所承担的迭代集合，无需任何运行期协调。其局限在于无法适应负载不均衡。

**`static, chunk`**：映射同样在进入循环时即可确定，运行期无需协调，因而开销依旧很低；轮转分配把工作量较大的迭代摊薄到各线程。对于本实验这类**工作量单调递增**的负载，该策略能以极低的运行期开销获得接近理想的负载均衡。

**`dynamic, chunk`**：线程完成当前块之后，才向运行时申请下一块。这种「先到先得」能适应任意分布，但每次申请都需要一次原子操作或加锁，块越小、申请次数越多、开销越大。

**`guided`**：初始块较大（约为剩余迭代数除以线程数），随后逐步缩小，最小不低于 `chunk_size`（默认为 1）。其设计意图是：前期用大块摊薄调度开销，后期用小块做精细均衡。

> **经验规则**：可先采用默认的 `static` 进行测量；若加速比明显低于线程数，再依次尝试 `static, chunk` 与 `guided`。仅当工作量分布随机且差异显著时，再考虑 `dynamic`。

### 2.5 `chunk_size` 的取舍

块大小同时影响两件事，方向相反：

| | 块小 | 块大 |
|---|---|---|
| 负载均衡度 | 好 | 差 |
| 调度开销 | 高 | 低 |
| 缓存局部性 | 差 | 好 |
| 伪共享风险 | 高（相邻迭代写相邻内存时） | 低 |

因此不存在一个普适的最优块大小，需要针对具体负载实测。第 12 节的练习 1 即为此设计。

## 3. OpenMP 关键知识点

### 3.1 `schedule(runtime)`：一次编译，多种策略

若将策略固定写在制导语句中，比较各种策略就需要分别编译多个版本，而不同版本之间可能因编译器优化差异而缺乏可比性。

`schedule(runtime)` 解决了这个问题：它把策略的决定推迟到运行时。本实验的内核只编译**一次**：

```c
#pragma omp parallel for num_threads(thread_count) \
    reduction(+ : approx) schedule(runtime)
for (long i = 0; i < n; i++) { ... }
```

运行时的实际策略由两种途径指定：

| 途径 | 写法 | 优先级 |
|---|---|---|
| 环境变量 | `export OMP_SCHEDULE="dynamic,4"` | 低 |
| 库函数 | `omp_set_schedule(omp_sched_dynamic, 4);` | 高（覆盖环境变量） |

本实验采用库函数，在同一次运行内依次切换五种调度配置：

```c
const omp_sched_t kind[NPOLICY] = {omp_sched_static, omp_sched_static,
                                   omp_sched_dynamic, omp_sched_guided,
                                   omp_sched_auto};
const int arg[NPOLICY] = {0, chunk, chunk, 0, 0};

for (int p = 0; p < NPOLICY; p++) {
  omp_set_schedule(kind[p], arg[p]);
  /* ... 计时并调用同一个内核 ... */
}
```

> `arg` 取 0 表示使用该调度类型的默认块大小。对 `static` 而言，未指定块大小时迭代空间被划分为大小大致相等的连续块，每个线程至多分得一块（GCC 的实现为 $\lceil n/p \rceil$）；对 `dynamic` 与 `guided` 而言，默认块大小均为 1。

### 3.2 迭代到线程的映射记录

为使「划分方式」这一抽象概念可被直接观察，内核在循环体内记录了每次迭代的执行线程：

```c
for (long i = 0; i < n; i++) {
  approx += f(i);
  if (map != NULL) {
    map[i] = omp_get_thread_num();
  }
}
```

该记录仅在 `n <= 64` 时启用（传入非空指针），因而不会干扰大规模下的性能测量。

### 3.3 线程亲和性：`OMP_PROC_BIND` 与 `OMP_PLACES`

调度配置决定「哪个线程执行哪些迭代」，线程亲和性则决定「哪个线程运行在哪个物理核心上」。两者是不同层次的问题。

| 环境变量 | 常用取值 | 含义 |
|---|---|---|
| `OMP_PROC_BIND` | `false` | 不绑定，允许操作系统迁移线程 |
| | `true` | 绑定，线程一旦分配便不再迁移 |
| | `close` | 绑定，且把线程放在彼此邻近的核心上 |
| | `spread` | 绑定，且把线程尽量分散到不同的插槽或簇 |
| `OMP_PLACES` | `cores` | 每个绑定位置为一个物理核心 |
| | `threads` | 每个绑定位置为一个硬件线程 |
| | `sockets` | 每个绑定位置为一个处理器插槽 |

**在 big.LITTLE 异构平台上，不绑定亲和性会带来两类问题**：

1. **测量不可重复**。同一份代码两次运行，若线程分别被分配至高性能核心与高能效核心，耗时可能相差 2 倍以上。
2. **调度配置的效果被掩盖**。假设 `static, chunk` 已经把工作量分配得相当均匀，但其中一个线程被分配到高能效核心上，它仍然会最后完成，于是均衡带来的收益被核心之间的速度差异抵消。

因此在异构平台上做性能实验，测速前应当固定亲和性：

```bash
export OMP_PROC_BIND=close
export OMP_PLACES=cores
export OMP_DISPLAY_ENV=TRUE   # 打印运行时配置，便于核对
```

> `OMP_DISPLAY_ENV=TRUE` 会在程序启动时把全部内部控制变量打印到标准错误，是核对实验环境是否符合预期的便捷手段。

## 4. 环境准备与检查

本节确认三项内容：编译器是否支持 OpenMP、运行时报告的处理器数量、以及各处理器核心的最大频率是否一致。

第三项检查针对**异构多核**平台。Arm 的 big.LITTLE 架构把高性能核心与高能效核心集成在同一块芯片上，二者的频率与微架构均不相同。在这类平台上，同一段代码在不同类型的核心上执行，耗时可能相差 2 倍以上，线程数与加速比之间因而不再是简单的线性关系。

需要说明的是，最大频率不一致只是异构多核的**必要非充分**证据：同构多核平台也可能因加速频率（boost）策略或芯片分级（binning）而上报不同的 `cpuinfo_max_freq`。因此下面的检查只给出提示，确认平台是否为异构架构还需结合 `lscpu` 输出的核心型号信息。

In [ ]:
import os
import re
import subprocess
import platform

print('=' * 60)
print(' 一、平台信息')
print('=' * 60)
print('操作系统   :', platform.system(), platform.release())
print('处理器架构 :', platform.machine())
print('逻辑核心数 :', os.cpu_count())

print()
print('=' * 60)
print(' 二、编译器与 OpenMP 支持')
print('=' * 60)
gcc_ver = subprocess.run(['gcc', '--version'], capture_output=True,
                         text=True).stdout.splitlines()[0]
print('编译器     :', gcc_ver)

probe = subprocess.run('echo | gcc -fopenmp -dM -E -x c - | grep _OPENMP',
                       shell=True, capture_output=True, text=True).stdout.strip()
if probe:
    ver = int(probe.split()[-1])
    spec = {200805: '3.0', 201107: '3.1', 201307: '4.0',
            201511: '4.5', 201811: '5.0', 202011: '5.1'}.get(ver, '未知')
    print('_OPENMP    :', ver, '(对应 OpenMP %s 规范)' % spec)
    print('数组段归约 :', '支持' if ver >= 201511 else '不支持（需要 4.5 及以上）')
else:
    print('⚠️  未检测到 OpenMP 支持，请确认编译时带有 -fopenmp')

print()
print('=' * 60)
print(' 三、核心频率与异构性检查')
print('=' * 60)
freqs = []
for cpu in range(os.cpu_count() or 1):
    path = '/sys/devices/system/cpu/cpu%d/cpufreq/cpuinfo_max_freq' % cpu
    try:
        with open(path) as f:
            freqs.append((cpu, int(f.read().strip()) // 1000))
    except OSError:
        pass

if not freqs:
    print('无法读取 cpufreq 节点，跳过异构性检查。')
else:
    for cpu, mhz in freqs:
        print('  CPU%-2d 最大频率: %5d MHz' % (cpu, mhz))
    distinct = sorted(set(m for _, m in freqs))
    if len(distinct) > 1:
        print()
        print('⚠️  检测到 %d 种不同的最大频率，本平台可能为异构多核架构（如 Arm big.LITTLE）。'
              % len(distinct))
        print('    请结合 lscpu 输出的核心型号信息进一步确认。')
        print('    若确为异构平台，测速前建议执行：')
        print('      export OMP_PROC_BIND=close')
        print('      export OMP_PLACES=cores')
    else:
        print()
        print('✅ 全部核心的最大频率一致，可按同构多核平台处理。')

print()
print('OMP_NUM_THREADS =', os.environ.get('OMP_NUM_THREADS', '（未设置，由运行时决定）'))
print('OMP_PROC_BIND   =', os.environ.get('OMP_PROC_BIND', '（未设置）'))
print('OMP_PLACES      =', os.environ.get('OMP_PLACES', '（未设置）'))

## 5. 实验工具函数

本节定义三个贯穿全章的辅助函数，后续各实验均直接调用，不再重复说明。

| 函数 | 作用 |
|---|---|
| `compile_c(src)` | 以 `-O3 -fopenmp -Wall -Wextra` 编译指定源文件，并回显全部告警 |
| `run_c(binary, *args)` | 运行可执行文件并原样打印其标准输出 |
| `parse_table(output)` | 从程序输出的结果表中提取「方法名 / 耗时 / 加速比 / 校验」四列 |

**关于编译选项**：全章统一使用 `-O3 -fopenmp`。AArch64 平台的 NEON 属于基线指令集，无需附加 `-march` 或 `-mcpu` 选项。`-Wall -Wextra` 用于暴露数据环境声明不当引发的告警，这类告警在 OpenMP 程序中往往是并发缺陷的征兆，不应忽略。

In [ ]:
import subprocess
import re
import os

SRC_DIR = 'src_for_schedule'
os.makedirs(SRC_DIR, exist_ok=True)

CFLAGS = ['-O3', '-fopenmp', '-Wall', '-Wextra']


def compile_c(src, extra=('-lm',)):
    """编译单个源文件，返回可执行文件路径；编译失败时抛出异常。"""
    src_path = os.path.join(SRC_DIR, src)
    binary = os.path.join(SRC_DIR, os.path.splitext(src)[0])
    cmd = ['gcc'] + CFLAGS + ['-o', binary, src_path] + list(extra)
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print(proc.stderr.rstrip())
    if proc.returncode != 0:
        raise RuntimeError('编译失败：%s' % src)
    print('✅ 编译通过，无告警' if not proc.stderr.strip()
          else '⚠️  编译通过，但存在告警，请逐条阅读')
    return binary


def run_c(binary, *args, env=None):
    """运行可执行文件，打印并返回其标准输出。"""
    cmd = [binary] + [str(a) for a in args]
    print('$', ' '.join(cmd))
    print()
    run_env = dict(os.environ)
    if env:
        run_env.update({k: str(v) for k, v in env.items()})
    proc = subprocess.run(cmd, capture_output=True, text=True, env=run_env)
    print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print('[stderr]', proc.stderr.rstrip())
    return proc.stdout


ROW_RE = re.compile(r'^\|\s*(.+?)\s*\|\s*([0-9.]+)\s*\|\s*([0-9.]+)x\s*\|\s*(\S+)\s*\|$')


def parse_table(output):
    """解析结果表，返回 [(方法名, 耗时ms, 加速比, 校验结论), ...]。"""
    rows = []
    for line in output.splitlines():
        m = ROW_RE.match(line.strip())
        if m:
            rows.append((m.group(1), float(m.group(2)),
                         float(m.group(3)), m.group(4)))
    return rows


print('工具函数已就绪，源码目录：', os.path.abspath(SRC_DIR))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

C_BASE, C_GOOD, C_FAIL, C_SLOW = '#7f7f7f', '#1f77b4', '#d62728', '#ff7f0e'


def plot_speedup(rows, title, figsize=(10, 5)):
    """绘制加速比柱状图。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。"""
    if not rows:
        print('未解析到结果行，请先运行上一单元格。')
        return
    names = [r[0] for r in rows]
    speeds = [r[2] for r in rows]
    colors = []
    for i, (_, _, sp, chk) in enumerate(rows):
        if i == 0:
            colors.append(C_BASE)
        elif chk == 'FAIL':
            colors.append(C_FAIL)
        elif sp < 1.0:
            colors.append(C_SLOW)
        else:
            colors.append(C_GOOD)

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(range(len(names)), speeds, color=colors,
                  edgecolor='black', linewidth=0.6, width=0.6)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Speedup vs. Serial Baseline')
    ax.set_title(title, fontsize=12, pad=12)
    ax.grid(axis='y', linestyle=':', alpha=0.5)
    ax.set_axisbelow(True)

    for bar, (_, ms, sp, chk) in zip(bars, rows):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02,
                '%.2fx\n%.1f ms%s' % (sp, ms, '' if chk in ('-', 'PASS') else '\nFAIL'),
                ha='center', va='bottom', fontsize=8)

    ax.set_ylim(0, max(speeds) * 1.30)
    plt.tight_layout()
    plt.show()


print('绘图函数已就绪。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。')

## 6. 版本设计总览

本实验与其余实验的组织方式不同：它只有**一个**并行内核，表格中的五行是同一个内核在五种调度配置下的表现。

| 行 | 调度配置 | 说明 |
|---|---|---|
| V0 | 串行基准 | 参照 |
| S1 | `schedule(static)` | 默认策略，连续大块 |
| S2 | `schedule(static, chunk)` | 轮转分配 |
| S3 | `schedule(dynamic, chunk)` | 先到先得 |
| S4 | `schedule(guided)` | 块长递减 |
| S5 | `schedule(auto)` | 交由实现决定 |

由于五行共用同一份编译产物、同一份数据、同一次进程，它们之间的差异可明确归因于调度配置本身。

## 7. 逐策略代码讲解

### 7.1 串行基准

```c
static double sum_serial(long n) {
  double approx = 0.0;
  for (long i = 0; i < n; i++) {
    approx += f(i);
  }
  return approx;
}
```

> **注意循环上界**：此处必须写作 `i < n`。若误写为 `i <= n`，将多执行一次迭代；而由于工作量随下标递增，`f(n)` 恰是工作量最大的一次调用，串行基准会因此显著偏慢，进而虚高全部并行配置的加速比。在工作量不均衡的负载中，边界条件的错误对性能数据的影响远大于均衡负载，这一点尤需留意。

### 7.2 并行内核

```c
static double sum_runtime(long n, int thread_count, int *map) {
  double approx = 0.0;

#pragma omp parallel for num_threads(thread_count) \
    reduction(+ : approx) schedule(runtime)
  for (long i = 0; i < n; i++) {
    approx += f(i);
    if (map != NULL) {
      map[i] = omp_get_thread_num();
    }
  }
  return approx;
}
```

该函数综合了前两个实验的全部要点：`reduction` 处理累加（实验二），迭代之间彼此独立、无循环携带依赖（实验三），`schedule(runtime)` 把划分策略留到运行时（本实验）。

### 7.3 关于 `map` 写入的开销

循环体内对 `map[i]` 的写入会引入一次分支与一次访存。但由于 `f(i)` 内部有 $i+1$ 次 `sin()` 调用，其耗时远高于这一次写入，因而对计时结果的影响可以忽略。此外，`map` 在整个循环中是循环不变量，编译器通常会通过**循环去条件化**（loop unswitching）把该判断提到循环之外，因而大规模测量时循环体内并不存在这一分支。

## 8. 源代码写入

In [ ]:
%%writefile {SRC_DIR}/omp_for_schedule.c
#define _POSIX_C_SOURCE 200809L

#include <math.h>
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#ifndef _OPENMP
#error "OpenMP is required. Please compile with -fopenmp."
#endif

#define NTIMES 3
#define MAX_THREADS 16
#define NPOLICY 5
// The iteration-to-thread map is only printed for small problems.
#define MAP_MAX 64
#define TOL 1e-6

#define BANNER "============================================================"
#define LINE "------------------------------------------------------------"

// ----------------------------------------------------------------------------
// Common helpers
// ----------------------------------------------------------------------------
static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

static int check_diff(const double *ref, const double *test, long n,
                      double tol) {
  for (long i = 0; i < n; i++) {
    if (fabs(ref[i] - test[i]) > tol) {
      return 0;
    }
  }
  return 1;
}

static void print_table_header(void) {
  printf("\n%s\n", LINE);
  printf("| %-26s | %9s | %7s | %-5s |\n", "Schedule", "Time(ms)", "Speedup",
         "Check");
  printf("|----------------------------|-----------|---------|-------|\n");
}

static void print_row(const char *name, double time_ms, double base_ms,
                      int check) {
  const char *status = (check < 0) ? "-" : (check ? "PASS" : "FAIL");
  double speedup = (time_ms > 0.0) ? base_ms / time_ms : 0.0;
  printf("| %-26s | %9.3f | %6.2fx | %-5s |\n", name, time_ms, speedup, status);
}

// One character per iteration, so the partition is visible at a glance.
static void print_map(const char *name, const int *map, long n) {
  printf("  %-22s ", name);
  for (long i = 0; i < n; i++) {
    int id = map[i];
    if (id < 0) {
      putchar('.');
    } else if (id < 10) {
      putchar((char)('0' + id));
    } else {
      putchar((char)('A' + id - 10));
    }
  }
  putchar('\n');
}

// ----------------------------------------------------------------------------
// Unbalanced workload: iteration i performs i + 1 calls to sin().
// ----------------------------------------------------------------------------
static double f(long i) {
  long first = i * (i + 1) / 2;
  long last = first + i;
  double val = 0.0;

  for (long j = first; j <= last; j++) {
    val += sin((double)j);
  }
  return val;
}

// ============================================================================
// V0: Serial baseline
// ============================================================================
static double sum_serial(long n) {
  double approx = 0.0;

  for (long i = 0; i < n; i++) {
    approx += f(i);
  }
  return approx;
}

// ============================================================================
// Parallel kernel. The policy is left to the runtime so that one binary can
// measure every schedule. Pass map = NULL to skip the ownership recording.
// ============================================================================
static double sum_runtime(long n, int thread_count, int *map) {
  double approx = 0.0;

#pragma omp parallel for num_threads(thread_count) reduction(+ : approx) \
    schedule(runtime)
  for (long i = 0; i < n; i++) {
    approx += f(i);
    if (map != NULL) {
      map[i] = omp_get_thread_num();
    }
  }
  return approx;
}

int main(int argc, char *argv[]) {
  if (argc != 4) {
    printf("Usage: %s <n> <thread_count> <chunk_size>\n", argv[0]);
    printf("Example: %s 10000 4 2\n", argv[0]);
    return 1;
  }

  long n = strtol(argv[1], NULL, 10);
  int thread_count = (int)strtol(argv[2], NULL, 10);
  int chunk = (int)strtol(argv[3], NULL, 10);

  if (n <= 0) {
    printf("Error: n must be > 0\n");
    return 1;
  }
  if (thread_count < 1 || thread_count > MAX_THREADS) {
    printf("Error: thread_count must be between 1 and %d\n", MAX_THREADS);
    return 1;
  }
  if (chunk < 1) {
    printf("Error: chunk_size must be >= 1\n");
    return 1;
  }

  printf("%s\n", BANNER);
  printf(" Lab 4: Loop Scheduling and Load Balancing\n");
  printf(" n: %ld | Threads: %d | Chunk: %d | Runs: %d\n", n, thread_count,
         chunk, NTIMES);
  printf(" _OPENMP: %d | Procs: %d\n", _OPENMP, omp_get_num_procs());
  printf("%s\n", BANNER);

  const omp_sched_t kind[NPOLICY] = {omp_sched_static, omp_sched_static,
                                     omp_sched_dynamic, omp_sched_guided,
                                     omp_sched_auto};
  const int arg[NPOLICY] = {0, chunk, chunk, 0, 0};
  char label[NPOLICY][32];

  snprintf(label[0], sizeof(label[0]), "schedule(static)");
  snprintf(label[1], sizeof(label[1]), "schedule(static, %d)", chunk);
  snprintf(label[2], sizeof(label[2]), "schedule(dynamic, %d)", chunk);
  snprintf(label[3], sizeof(label[3]), "schedule(guided)");
  snprintf(label[4], sizeof(label[4]), "schedule(auto)");

  int record_map = (n <= MAP_MAX);
  int *maps[NPOLICY] = {NULL};
  if (record_map) {
    for (int p = 0; p < NPOLICY; p++) {
      maps[p] = (int *)malloc((size_t)n * sizeof(int));
      if (maps[p] == NULL) {
        printf("Error: memory allocation failed\n");
        return 1;
      }
      for (long i = 0; i < n; i++) {
        maps[p][i] = -1;
      }
    }
  }

  double start = 0.0;
  double t_serial = 0.0;
  double res_serial = 0.0;
  double t[NPOLICY] = {0.0};
  double res[NPOLICY] = {0.0};

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    res_serial = sum_serial(n);
    t_serial += get_time_ms() - start;
  }
  t_serial /= NTIMES;

  for (int p = 0; p < NPOLICY; p++) {
    omp_set_schedule(kind[p], arg[p]);
    for (int r = 0; r < NTIMES; r++) {
      start = get_time_ms();
      res[p] = sum_runtime(n, thread_count, maps[p]);
      t[p] += get_time_ms() - start;
    }
    t[p] /= NTIMES;
  }

  print_table_header();
  print_row("V0: Serial Baseline", t_serial, t_serial, -1);
  for (int p = 0; p < NPOLICY; p++) {
    print_row(label[p], t[p], t_serial,
              check_diff(&res_serial, &res[p], 1, TOL));
  }
  printf("%s\n", LINE);

  printf("\nResult (V0): %.6f\n", res_serial);

  if (record_map) {
    printf("\nIteration to thread map (one character per iteration):\n");
    for (int p = 0; p < NPOLICY; p++) {
      print_map(label[p], maps[p], n);
    }
  } else {
    printf("\nHint: run with n <= %d to print the iteration to thread map.\n",
           MAP_MAX);
  }

  for (int p = 0; p < NPOLICY; p++) {
    free(maps[p]);
  }
  return 0;
}

## 9. 编译与运行

### 9.1 编译

In [ ]:
bin_sched = compile_c('omp_for_schedule.c')

### 9.2 小规模运行：观察划分方式

先以 $n = 40$、4 线程、块长 2 运行。该规模下程序会打印迭代到线程的字符映射图，每个字符代表一次迭代，字符本身是执行该迭代的线程编号。

请重点比较 `static` 与 `static, 2` 两行的形态差异。

In [ ]:
out_map = run_c(bin_sched, 40, 4, 2)

### 9.3 映射图解读

预期形态如下（实际输出可能因实现而略有不同）：

```text
  schedule(static)       0000000000111111111122222222223333333333
  schedule(static, 2)    0011223300112233001122330011223300112233
  schedule(dynamic, 2)   3311002233110022331100223311002233110022
  schedule(guided)       3333333333111111110000002222333222110303
  schedule(auto)         0000000000111111111122222222223333333333
```

**`static`**：四个连续大块。由于工作量随下标递增，线程 3 所承担的后四分之一迭代，其总工作量远高于线程 0。

**`static, 2`**：以 2 为单位轮转。每个线程都同时分到了工作量较小的迭代（小下标）与工作量较大的迭代（大下标），因而各线程的总工作量自然趋于接近。

**`dynamic` 与 `guided`**：这两行的形态取决于运行时的实际竞争情况。在核心数充足时会呈现出不规则的交错；若可用核心只有一个，则往往由第一个线程连续取走全部块，整行显示为同一个数字——这本身也说明了动态调度「先到先得」的性质。

**`auto`**：在本实验所用的 GCC 版本中，`auto` 的实际行为等同于 `static`，因此该行通常与第一行完全相同。需要说明的是，规范并未规定 `auto` 的具体策略，上述行为是实现自行选择的结果，其他编译器或后续版本可能给出不同的映射（思考题 4 即针对这一点）。`auto` 并不保证「智能选择」，它只是把策略的决定权交给具体实现，而实现完全可以采用最简单的策略。

### 9.4 大规模运行：性能测量

以 $n = 10000$、4 线程、块长 2 运行。此规模下总工作量约为 $5\times10^7$ 次 `sin()` 调用，串行耗时在秒级，足以让调度策略的差异显现。

In [ ]:
out_sched = run_c(bin_sched, 10000, 4, 2)
rows_sched = parse_table(out_sched)
print()
for name, ms, sp, chk in rows_sched:
    print('  %-28s %9.3f ms  %5.2fx  %s' % (name, ms, sp, chk))

## 10. 结果可视化

灰色为串行基准。理论加速比上限已在 1.3 节推导：在 4 线程、三角形负载、连续块划分下约为 2.3；而均衡良好的配置可望接近 4。

In [ ]:
plot_speedup(rows_sched,
             'Lab 4: Scheduling Policies (n = 10000, 4 threads, chunk = 2)',
             figsize=(11, 5))

## 11. 结果分析

> 以下结论针对**趋势规律**。具体数值随平台、核心数与线程亲和性设置而变化。

**① `schedule(static)` 的加速比明显低于线程数**

这与 1.3 节的推导一致：连续块划分使最后一个线程承担了约 1.75 倍于平均值的工作量，而隐式栅栏让整体耗时由它决定。**这一损失与线程的执行速度无关，而完全由划分方式引起。**

**② `schedule(static, 2)` 通常给出最佳或接近最佳的结果**

对于工作量**单调变化**的负载，轮转分配是开销较低的均衡方式：映射在进入循环时即已确定，运行期无需协调，却能把工作量较大的迭代均匀分配给各线程。

**③ `schedule(dynamic, 2)` 均衡最好，但开销也最高**

块长为 2 意味着 $n/2 = 5000$ 次运行时申请，每次申请都需要一次原子操作。在本实验中，由于单次迭代的计算量较大，这一开销仍被摊薄；若把 `f(i)` 换成一次简单的加法，`dynamic, 2` 会显著慢于 `static`。

**④ `schedule(guided)` 是折中方案**

它以较大的初始块摊薄前期开销，再以逐步缩小的块处理尾部。在工作量分布未知时，`guided` 通常是比 `dynamic` 更稳妥的默认选择。

**⑤ `schedule(auto)` 与 `schedule(static)` 表现一致**

映射图已经直接印证了这一点：两行字符完全相同。结论是：**不宜依赖 `auto` 自动完成优化**。其价值在于为具体实现保留针对硬件进行适配的空间，而当前 GCC 尚未利用这一机制。

**⑥ 关于异构核心的干扰**

若在 big.LITTLE 平台上未设置 `OMP_PROC_BIND`，上述规律可能被核心速度差异所掩盖：即便工作量分配均匀，被调度到高能效核心上的线程仍可能最后完成。此时观察到的将主要是线程与核心之间映射的随机性，而非调度配置本身的差异。第 12 节的练习 3 专门用于验证这一点。

## 12. 🔧 动手练习

**练习 1**　固定 $n = 10000$、4 线程，把块长依次取 1、2、8、32、128、512，观察 `static, chunk` 与 `dynamic, chunk` 的耗时随块长的变化。请指出两条曲线的形状差异，并解释原因。

**练习 2**　把 `f(i)` 改为工作量与 $i$ **无关**的形式（例如固定执行 100 次 `sin()`），重新比较四种策略。在均衡负载下，哪种策略最快？为什么？

**练习 3**　分别在设置与不设置 `OMP_PROC_BIND=close` 的条件下各运行五次，比较两组结果的**波动幅度**。

**练习 4**　把线程数依次取 1、2、4、8，观察 `static` 的加速比是否符合 1.3 节推导所给出的趋势（一般表达式的推导见思考题 1）。

**练习 5**（进阶）　把内核中的 `schedule(runtime)` 改为 `schedule(static)` 后重新编译，再通过 `OMP_SCHEDULE` 环境变量尝试切换策略。记录观察到的现象，并解释 `runtime` 与其余取值在语义上的区别。

### 12.1 练习 1 的脚手架：块大小扫描

In [ ]:
import matplotlib.pyplot as plt

chunks = [1, 2, 8, 32, 128, 512]
N_FIX, NT = 10000, 4
stat_ms, dyn_ms = [], []

for c in chunks:
    out = subprocess.run([bin_sched, str(N_FIX), str(NT), str(c)],
                         capture_output=True, text=True).stdout
    rows = parse_table(out)
    byname = {r[0]: r for r in rows}
    s = byname.get('schedule(static, %d)' % c)
    d = byname.get('schedule(dynamic, %d)' % c)
    stat_ms.append(s[1] if s else float('nan'))
    dyn_ms.append(d[1] if d else float('nan'))
    print('chunk = %-4d  static %8.1f ms   dynamic %8.1f ms'
          % (c, stat_ms[-1], dyn_ms[-1]))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(chunks, stat_ms, marker='o', label='schedule(static, chunk)')
ax.plot(chunks, dyn_ms, marker='s', label='schedule(dynamic, chunk)')
ax.set_xscale('log', base=2)
ax.set_xlabel('chunk size (log2 scale)')
ax.set_ylabel('Time (ms)')
ax.set_title('Lab 4: Time vs. Chunk Size (n = 10000, 4 threads)')
ax.set_xticks(chunks)
ax.set_xticklabels([str(c) for c in chunks])
ax.grid(linestyle=':', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

### 12.2 练习 3 的脚手架：线程亲和性对波动的影响

In [ ]:
import statistics

for label, env in (('未绑定亲和性', {}),
                   ('OMP_PROC_BIND=close', {'OMP_PROC_BIND': 'close',
                                            'OMP_PLACES': 'cores'})):
    samples = []
    run_env = dict(os.environ)
    run_env.update(env)
    for _ in range(5):
        out = subprocess.run([bin_sched, '4000', '4', '2'],
                             capture_output=True, text=True,
                             env=run_env).stdout
        rows = parse_table(out)
        static_row = [r for r in rows if r[0] == 'schedule(static)']
        if static_row:
            samples.append(static_row[0][1])
    if len(samples) >= 2:
        spread = (max(samples) - min(samples)) / statistics.mean(samples)
        print('%-22s 均值 %8.2f ms   极差占比 %5.1f%%'
              % (label, statistics.mean(samples), spread * 100))
print()
print('在异构多核平台上，绑定亲和性后极差占比通常明显下降。')
print('若本机为同构平台或核心数为 1，两组结果不会有系统性差异。')

## 13. 🤔 思考题

**思考题 1**　1.3 节推导出 4 线程、三角形负载、`static` 划分下的加速比上限约为 2.3。请把该推导推广到 $p$ 个线程，给出上限的一般表达式，并说明当 $p \to \infty$ 时，该上限与 $p$ 的比值（即并行效率的上限）趋于多少。这一结果说明了什么？

**思考题 2**　`schedule(dynamic, 1)` 的均衡性最好，为什么实践中很少使用它？请从调度开销与缓存局部性两方面作答。

**思考题 3**　假设某循环的迭代之间存在**缓存复用**（第 $i$ 次迭代访问的数据与第 $i+1$ 次高度重叠），此时 `static` 与 `dynamic` 哪一个更有利？为什么？

**思考题 4**　本实验的映射图显示 GCC 把 `auto` 实现为 `static`。若换用另一款编译器，映射图可能不同。这是否意味着使用 `auto` 的程序不可移植？请说明「可移植」在此处的确切含义。

**思考题 5**　`guided` 的初始块长约为「剩余迭代数 ÷ 线程数」。请说明为何这一取法既能摊薄开销、又能保证尾部足够细，并推算 $n = 10000$、$p = 4$ 时前三个块的大致长度。

**思考题 6**（综合）　调度策略解决的是「工作量不均」，线程亲和性解决的是「核心速度不均」。在 big.LITTLE 平台上，若两类不均同时存在，仅靠调整 `schedule` 能否完全消除损失？若不能，还需要什么手段？

## 14. 📌 本实验小结

| 概念 | 要点 |
|---|---|
| 负载不均衡 | 隐式栅栏使整体耗时由最慢线程决定 |
| `static` | 零开销，仅适用于均衡负载 |
| `static, chunk` | 轮转分配，零运行期开销，适合单调变化的负载 |
| `dynamic, chunk` | 先到先得，适应任意分布，调度开销较高 |
| `guided` | 块长递减，开销与均衡的折中 |
| `auto` | 交由实现决定，GCC 当前等同于 `static` |
| `schedule(runtime)` | 一次编译、多种策略，便于对照实验 |
| `omp_set_schedule` | 优先级高于 `OMP_SCHEDULE` 环境变量 |
| 线程亲和性 | `OMP_PROC_BIND` / `OMP_PLACES`，异构平台上应当设置 |

### 调度配置的选择次序

1. 默认 `static`，测量加速比；
2. 若明显低于线程数，检查工作量分布是否单调 → 试 `static, chunk`；
3. 分布随机且差异悬殊 → 试 `guided`；
4. 仍不理想且单次迭代计算量足够大 → 试 `dynamic, chunk`；
5. 全程固定线程亲和性，否则测量结果不可重复。

### 与后续实验的衔接

本实验讨论的是**并行区域内部**的开销。下一个实验把视角移到并行区域**本身**：当一个算法需要反复进入并行区域时，创建与销毁线程组的代价会累积到何种程度，又该如何规避。